In [1]:
from db_connection import setup_sakila, save_result_csv

engine = setup_sakila(displaylimit=None)

displaylimit: Value None will be treated as 0 (no limit)

## 1. L字型テーブル

部署情報と社員情報を1つのテーブルに保存すると、社員がいない部署では社員関連の列が`NULL`になる。

| 部署番号 | 部署名 | 部署所在地 | 社員番号 | 社員名 | 役職 | 入社日 |
|---:|---|---|---:|---|---|---|
| 5 | 人事部 | 横浜 | 1001 | 千尋 | 社員 | 2015-10-21 |
| 2 | IT部 | 大阪 | NULL | NULL | NULL | NULL |
| 3 | 広報部 | 名古屋 | NULL | NULL | NULL | NULL |
| 4 | 会計部 | 札幌 | 1002 | ハウル | 管理者 | 2015-10-21 |
| 1 | 管理部 | 福岡 | NULL | NULL | NULL | NULL |

#### ※ NULLとは

`NULL`は、行を登録する際に値が入力されていない状態を表す。

#### ! NULLが多い場合の問題

- データ量が多いと、不要な`NULL`によって保存領域が無駄になる。
- データ管理や検索が複雑になり、データ品質にも影響する。

> 保存領域の効率・管理のしやすさ・データ品質を高めるため、必要に応じてテーブルを分割する。

#### # データ収集時

- データ収集段階では作業効率を優先し、`NULL`や不完全な値を許容する場合もある。

```text
データ収集
→ 前処理
→ モデリング（テーブル分割・正規化）
```

### ※ `L字型テーブル`：テーブル内で`NULL`値を持つ行（データ）を上側に配置した形。

#### # 部署テーブルのL字型テーブル

| 部署番号 | 部署名 | 部署所在地 | 社員番号 | 社員名 | 社員役職 | 入社日 |
|---:|---|---|---:|---|---|---|
| 1 | 管理部 | 東京 | NULL | NULL | NULL | NULL |
| 2 | IT部 | 大阪 | NULL | NULL | NULL | NULL |
| 3 | 広報部 | 名古屋 | NULL | NULL | NULL | NULL |
| 4 | 会計部 | 京都 | 1002 | ハウル | 管理者 | 2015-10-21 |
| 5 | 人事部 | 横浜 | 1001 | 千尋 | 社員 | 2015-10-21 |

## 2. NULL値を持つカラムを基準にテーブルを分割

#### # 部署情報と社員情報を分離し、不要な`NULL`を減らす。

#### 2-1. 部署情報と社員情報を分離

##### 1) 部署テーブル

| 部署番号 | 部署名 | 部署所在地 |
|---:|---|---|
| 1 | 管理部 | 東京 |
| 2 | IT部 | 大阪 |
| 3 | 広報部 | 名古屋 |
| 4 | 会計部 | 京都 |
| 5 | 人事部 | 横浜 |

#### 2) 社員テーブル

| 社員番号 | 社員名 | 役職 | 入社日 |
|---:|---|---|---|
| 1001 | ハウル | 管理者 | 2015-10-21 |
| 1002 | 千尋 | 社員 | 2015-10-21 |

カラムを基準にテーブルを分割すると、不要な`NULL`を減らし、データを効率的に保存できる。

→ 分割したテーブル間には`リレーション（Relation）`が必要になる。

#### 2-2. 分割したテーブルを関連付ける

テーブルを分割した場合、`親テーブルの主キー（Primary Key）`と`子テーブルの外部キー（Foreign Key）`を使って関連付ける。

#### 1) 部署テーブル（親テーブル）

| 部署番号 | 部署名 | 部署所在地 |
|---:|---|---|
| 1 | 管理部 | 東京 |
| 2 | IT部 | 大阪 |
| 3 | 広報部 | 名古屋 |
| 4 | 会計部 | 京都 |
| 5 | 人事部 | 横浜 |

#### 2) 社員テーブル（子テーブル）

社員テーブルの`部署番号`は`外部キー（Foreign Key）`である。

| 社員番号 | 社員名 | 社員役職 | 入社日 | 部署番号 |
|---:|---|---|---|---:|
| 1001 | ハウル | 管理者 | 2015-10-21 | 4 |
| 1002 | 千尋 | 社員 | 2015-10-21 | 5 |

```text
部署テーブル.部署番号（主キー）
        ↑
        │ 参照
        │
社員テーブル.部署番号（外部キー）
```

#### # 参照整合性

子テーブルが`外部キー（Foreign Key）`で親テーブルのデータを参照している場合、   
親テーブルの`参照されている値は変更・削除できない`。

#### # CASCADEオプション

親テーブルの`参照されている値を変更・削除`すると、  
`子テーブル`の参照しているデータも`変更・削除`されるオプション。

## 3. L字型テーブルの実習

#### 1) リレーションを1つの部署テーブルとして設計（L字型テーブル）

- 1つのテーブルにすべての情報を保存
- `NULL`が多く発生
- 保存領域を浪費
- データの重複や異常（Anomaly）が発生する可能性がある

In [2]:
%%sql

CREATE TABLE department_raw
(
    dept_no     INT,
    dept_name   VARCHAR(20),
    dept_loc    VARCHAR(20),

    emp_no      INT,
    emp_name    VARCHAR(20),
    emp_job     VARCHAR(20),
    hire_date   DATE
);

++
||
++
++

In [3]:
%%sql

INSERT INTO department_raw
VALUES
(1, '管理部', '東京', NULL, NULL, NULL, NULL),
(2, 'IT部', '大阪', NULL, NULL, NULL, NULL),
(3, '広報部', '名古屋', NULL, NULL, NULL, NULL),
(4, '会計部', '京都', 1001, 'ハウル', '管理者', '2015-10-21'),
(5, '人事部', '横浜', 1002, '千尋', '社員', '2015-10-21');

++
||
++
++

#### 2) リレーションを部署・社員の2つのテーブルに分割して設計

- `NULL`が発生するカラムを基準にテーブルを分割
- 重複を削除し、保存領域を節約
- `department.dept_no` → **主キー（Primary Key）**
- `employee.dept_no` → **外部キー（Foreign Key）**、2つのテーブルは`1:N関係`

In [6]:
%%sql

CREATE TABLE department
(
    dept_no     INT PRIMARY KEY,
    dept_name   VARCHAR(20) NOT NULL,
    dept_loc    VARCHAR(20) NOT NULL
);

++
||
++
++

In [7]:
%%sql

INSERT INTO department
VALUES
(1, '管理部', '東京'),
(2, 'IT部', '大阪'),
(3, '広報部', '名古屋'),
(4, '会計部', '京都'),
(5, '人事部', '横浜');

++
||
++
++

In [9]:
%%sql

CREATE TABLE employee
(
    emp_no      INT PRIMARY KEY,
    emp_name    VARCHAR(20) NOT NULL,
    emp_job     VARCHAR(20),
    hire_date   DATE,

    dept_no     INT,

    CONSTRAINT fk_employee_department
        FOREIGN KEY (dept_no)
        REFERENCES department(dept_no)
);

++
||
++
++

In [10]:
%%sql

INSERT INTO employee
VALUES
(1001, 'ハウル', '管理者', '2015-10-21', 4),
(1002, '千尋', '社員', '2015-10-21', 5);

++
||
++
++

#### # 保存領域の比較

- `department_raw`：**35**個のセルが必要

In [13]:
%%sql

SELECT * 
FROM department_raw;

dept_no,dept_name,dept_loc,emp_no,emp_name,emp_job,hire_date
1,管理部,東京,None,None,None,None
2,IT部,大阪,None,None,None,None
3,広報部,名古屋,None,None,None,None
4,会計部,京都,1001,ハウル,管理者,2015-10-21
5,人事部,横浜,1002,千尋,社員,2015-10-21


- `department` + `employee`：**25**個のセルが必要<br>
→ **10個のセル領域を節約**

In [14]:
%%sql

SELECT * 
FROM department;

dept_no,dept_name,dept_loc
1,管理部,東京
2,IT部,大阪
3,広報部,名古屋
4,会計部,京都
5,人事部,横浜


In [15]:
%%sql

SELECT * 
FROM employee;

emp_no,emp_name,emp_job,hire_date,dept_no
1001,ハウル,管理者,2015-10-21,4
1002,千尋,社員,2015-10-21,5


ただし、全体を照会する場合は`JOIN`が必要であり、`JOIN`を使用しない場合よりパフォーマンスが低下する可能性がある。

In [19]:
%%sql

SELECT
    d.dept_no,
    d.dept_name,
    d.dept_loc,
    e.emp_no,
    e.emp_name,
    e.emp_job,
    e.hire_date
FROM department d
LEFT JOIN employee e
ON d.dept_no = e.dept_no
ORDER BY d.dept_no;

dept_no,dept_name,dept_loc,emp_no,emp_name,emp_job,hire_date
1,管理部,東京,None,None,None,None
2,IT部,大阪,None,None,None,None
3,広報部,名古屋,None,None,None,None
4,会計部,京都,1001,ハウル,管理者,2015-10-21
5,人事部,横浜,1002,千尋,社員,2015-10-21
